# E2 confirmatory inference preflight

**TL;DR.** This deterministic preflight sends a complete fabricated 240-observation factorial
through the production confirmatory API. It asserts the frozen 36-bound, eight-paired-seed,
100,000-resample one-sided Bonferroni contract and demonstrates that one missing row fails closed.
All values are synthetic protocol fixtures; none are empirical or paper evidence.

## Context, assumptions, and pass criteria

- The independent unit is a paired training seed after equal AE/beta-VAE averaging within a
  domain; frames and model families are not treated as independent replicates.
- The proposed profile arm is compared with four frozen comparators over three domains and three
  endpoints, yielding exactly 36 elementary upper bounds.
- Primary UCBs are the nonstudentized percentile quantile at `1 - 0.05 / 36`, from 100,000
  paired-seed bootstrap replicates with seed 20260822. Holm is not part of this primary preflight.
- The fixture passes by construction. Removing any required row must raise before a result exists.

In [1]:
from itertools import product
from pathlib import Path
import sys

from IPython.display import display

ROOT = Path.cwd()
if ROOT.name == "notebooks":
    ROOT = ROOT.parent
elif (ROOT / "LatentSafety" / "src").exists():
    ROOT = ROOT / "LatentSafety"
if not (ROOT / "src" / "latent_safety").exists():
    raise RuntimeError(f"Could not locate repository root from {Path.cwd()}")
sys.path.insert(0, str(ROOT / "src"))

from latent_safety.analysis import (
    FROZEN_CONFIRMATORY_SPEC,
    ConfirmatoryObservation,
    run_confirmatory_inference,
)

spec = FROZEN_CONFIRMATORY_SPEC
spec.validate()
display({
    "status": "SYNTHETIC PROTOCOL PREFLIGHT — NOT EMPIRICAL EVIDENCE",
    "domains": spec.domains,
    "comparators": spec.comparators,
    "endpoints": spec.endpoints,
    "paired_seeds": spec.paired_seeds,
    "elementary_bounds": spec.elementary_bound_count,
    "bootstrap_resamples": spec.bootstrap_resamples,
    "bootstrap_seed": spec.bootstrap_seed,
    "primary_ucb_probability": spec.ucb_probability,
    "primary_ucb_method": spec.bootstrap_method,
    "multiplicity": spec.multiplicity_method,
})
assert spec.elementary_bound_count == 36
assert spec.bootstrap_resamples == 100_000
assert spec.bootstrap_seed == 20_260_822
assert abs(spec.ucb_probability - (1.0 - 0.05 / 36.0)) <= 1e-15

{'status': 'SYNTHETIC PROTOCOL PREFLIGHT — NOT EMPIRICAL EVIDENCE',
 'domains': ('controlled_cart_video',
  'controlled_pendulum_video',
  'controlled_dubins_navigation_pixels'),
 'comparators': ('none',
  'h_prediction',
  'fcsrl_feasibility_loss_adaptation',
  'append_true_h_none_view'),
 'endpoints': ('safety', 'reconstruction', 'rollout'),
 'paired_seeds': (100, 101, 102, 103, 104, 105, 106, 107),
 'elementary_bounds': 36,
 'bootstrap_resamples': 100000,
 'bootstrap_seed': 20260822,
 'primary_ucb_probability': 0.9986111111111111,
 'primary_ucb_method': 'one-sided nonstudentized percentile UCB',
 'multiplicity': 'Bonferroni over all 36 elementary bounds'}

## Data — complete synthetic factorial

In [2]:
observations = []
for domain_index, domain in enumerate(spec.domains):
    for family_index, family in enumerate(spec.model_families):
        for arm, seed in product((spec.proposed_arm, *spec.comparators), spec.paired_seeds):
            seed_index = spec.paired_seeds.index(seed)
            nuisance = 0.001 * domain_index + 0.0005 * family_index + 0.0001 * seed_index
            proposed = arm == spec.proposed_arm
            observations.append(
                ConfirmatoryObservation(
                    domain=domain,
                    model_family=family,
                    arm=arm,
                    seed=seed,
                    safety=(0.20 if proposed else 0.35) + nuisance,
                    reconstruction=(1.02 if proposed else 1.00) * (1.0 + nuisance),
                    rollout=(1.03 if proposed else 1.00) * (1.0 + nuisance),
                    run_complete=True,
                    coverage_complete=True,
                )
            )

assert len(observations) == 240
assert len({(row.domain, row.model_family, row.arm, row.seed) for row in observations}) == 240
display({
    "fixture_rows": len(observations),
    "fixture_status": "complete synthetic factorial",
    "run_complete": all(row.run_complete for row in observations),
    "coverage_complete": all(row.coverage_complete for row in observations),
})

{'fixture_rows': 240,
 'fixture_status': 'complete synthetic factorial',
 'run_complete': True,
 'coverage_complete': True}

## Results — exact primary contract and fail-closed rejection

In [3]:
result = run_confirmatory_inference(observations)
bound_preview = [
    {
        "hypothesis_id": bound.hypothesis_id,
        "point_mean": bound.point_mean,
        "upper_confidence_bound": bound.upper_confidence_bound,
        "passed": bound.passed,
    }
    for bound in result.bounds[:4]
]
display({
    "status": "SYNTHETIC API REGRESSION — NOT A RESULT",
    "input_observation_count": result.input_observation_count,
    "expected_observation_count": result.expected_observation_count,
    "bound_count": len(result.bounds),
    "safety_bound_count": result.safety_bound_count,
    "utility_bound_count": result.utility_bound_count,
    "passed_bound_count": result.passed_bound_count,
    "confirmatory_success_on_constructed_fixture": result.confirmatory_success,
    "bounded_preview": bound_preview,
})
assert result.input_observation_count == result.expected_observation_count == 240
assert len(result.bounds) == 36
assert result.safety_bound_count == 12
assert result.utility_bound_count == 24
assert result.all_elementary_bounds_passed

try:
    run_confirmatory_inference(observations[:-1])
except ValueError as error:
    rejection = str(error)
else:
    raise AssertionError("an incomplete confirmatory factorial was accepted")
assert "missing 1 cells" in rejection
display({
    "fail_closed_check": "passed",
    "removed_rows": 1,
    "rejection": rejection,
})

{'status': 'SYNTHETIC API REGRESSION — NOT A RESULT',
 'input_observation_count': 240,
 'expected_observation_count': 240,
 'bound_count': 36,
 'safety_bound_count': 12,
 'utility_bound_count': 24,
 'passed_bound_count': 36,
 'confirmatory_success_on_constructed_fixture': True,
 'bounded_preview': [{'hypothesis_id': 'controlled_cart_video::none::safety',
   'point_mean': -0.14999999999999997,
   'upper_confidence_bound': -0.14999999999999994,
   'passed': True},
  {'hypothesis_id': 'controlled_cart_video::none::reconstruction',
   'point_mean': 0.020000000000000004,
   'upper_confidence_bound': 0.020000000000000042,
   'passed': True},
  {'hypothesis_id': 'controlled_cart_video::none::rollout',
   'point_mean': 0.030000000000000034,
   'upper_confidence_bound': 0.03000000000000006,
   'passed': True},
  {'hypothesis_id': 'controlled_cart_video::h_prediction::safety',
   'point_mean': -0.14999999999999997,
   'upper_confidence_bound': -0.14999999999999994,
   'passed': True}]}

{'fail_closed_check': 'passed',
 'removed_rows': 1,
 'rejection': "incomplete confirmatory factorial: missing 1 cells; first=('controlled_dubins_navigation_pixels', 'beta_vae', 'append_true_h_none_view', 107)"}

## Takeaways

The production engine is runnable before unblinding, emits exactly the predeclared 36 bounds, and
rejects incomplete coverage. Passing this deliberately favorable fixture validates plumbing only;
it says nothing about a learned profile arm, any domain effect, or confirmatory success on real
final-test data.